# Monitoring and Alerts

[Read the step-by-step guide](https://docs.futureagi.com/docs/cookbook/quickstart/monitoring-alerts). Send ten synthetic support requests through an instrumented RAG agent. Then use the dashboard to examine latency and configure an alert. Use a fresh Python 3.12 kernel.

## Install

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fi-instrumentation-otel==1.5.1", "traceai-openai==0.1.16", "openai==1.109.1"], check=True)

In [ ]:
import os
from getpass import getpass

for name in ("FI_API_KEY", "FI_SECRET_KEY", "OPENAI_API_KEY"):
    if not os.environ.get(name):
        value = getpass(f"{name}: ")
        if not value:
            raise ValueError(f"{name} is required")
        os.environ[name] = value
        del value

# For a self-hosted collector, set FI_TRACES_URL before running this cell.
os.environ.setdefault("FI_TRACES_URL", "https://api.futureagi.com")

## Build and instrument a multi-step RAG agent

Register tracing before creating the OpenAI client so the nested model call is captured.

In [ ]:
import time
from openai import OpenAI
from fi_instrumentation import register, FITracer, using_user, using_session, using_metadata, using_tags
from fi_instrumentation.fi_types import ProjectType
from traceai_openai import OpenAIInstrumentor

# 1. Register tracing
trace_provider = register(
    project_type=ProjectType.OBSERVE,
    project_name="monitoring-demo",
    verbose=False,
)
OpenAIInstrumentor().instrument(tracer_provider=trace_provider)

client = OpenAI()
tracer = FITracer(trace_provider.get_tracer(__name__))

# 2. Define agent components using decorators

@tracer.tool(name="search_knowledge_base", description="Search product docs for relevant passages")
def search_knowledge_base(query: str) -> list[str]:
    """Simulates a vector DB search over product documentation."""
    knowledge = {
        "return": ["Items can be returned within 30 days.", "Refunds are processed in 5-7 business days."],
        "refund": ["Items can be returned within 30 days.", "Refunds are processed in 5-7 business days."],
        "shipping": ["Standard shipping takes 5-7 days.", "Express shipping is 1-2 business days.", "Free shipping on orders over $50."],
        "warranty": ["All electronics have a 1-year warranty.", "Extended warranty available for $29.99."],
        "pricing": ["The gear rental plan costs $49 per month.", "Annual billing saves 20%."],
        "plan": ["The gear rental plan costs $49 per month.", "Annual billing saves 20%."],
        "billing": ["The gear rental plan costs $49 per month.", "Annual billing saves 20%."],
        "account": ["Reset password via Settings → Security.", "Two-factor authentication is recommended."],
        "password": ["Reset password via Settings → Security.", "Two-factor authentication is recommended."],
    }
    results = []
    for key, docs in knowledge.items():
        if key in query.lower():
            results.extend(docs)
    if not results:
        results = ["Please visit our help center at help.example.com for more information."]
    return results


@tracer.chain(name="generate_response")
def generate_response(query: str, context_docs: list[str]) -> str:
    """Uses retrieved context to generate a grounded answer."""
    context = "\n".join(f"- {doc}" for doc in context_docs)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "You answer support questions for Ridgeline Outfitters. Answer using ONLY the provided context. "
                    "If the context does not contain the answer, say so.\n\n"
                    f"Context:\n{context}"
                ),
            },
            {"role": "user", "content": query},
        ],
    )
    return response.choices[0].message.content


@tracer.agent(name="support_rag_agent")
def support_rag_agent(query: str) -> str:
    """Top-level agent: retrieves docs then generates a grounded response."""
    docs = search_knowledge_base(query)
    answer = generate_response(query, docs)
    return answer

## Generate trace data across multiple queries

Each query makes one OpenAI request. The final flush sends buffered spans to your collector.

In [ ]:
# Diverse queries that exercise different knowledge base paths
test_queries = [
    "What is your return policy?",
    "How long does shipping take?",
    "Do you offer express shipping?",
    "What warranty comes with electronics?",
    "How much is the gear rental plan?",
    "Can I get a discount on annual billing?",
    "How do I reset my password?",
    "What is the refund timeline?",
    "Is there free shipping?",
    "Tell me about the extended warranty.",
]

users = ["user-alice", "user-bob", "user-carol", "user-dave", "user-eve"]
environments = ["production", "staging"]

print("Generating trace data...\n")
for i, query in enumerate(test_queries):
    user_id = users[i % len(users)]
    session_id = f"session-{user_id}-{i // len(users)}"
    env_tag = environments[i % len(environments)]

    with (
        using_user(user_id),
        using_session(session_id),
        using_metadata({"environment": env_tag, "query_index": str(i)}),
        using_tags([env_tag, "rag-pipeline", "monitoring-demo"]),
    ):
        answer = support_rag_agent(query)
        print(f"[{user_id}] Q: {query}")
        print(f"         A: {answer[:80]}...\n")

    # Small delay between queries to spread data points over time
    time.sleep(0.5)

if not trace_provider.force_flush():
    raise RuntimeError("Timed out waiting for spans to flush")
print("Finished sending the trace batch.")

## Analyze historical trends in the Charts tab

Open **Tracing → monitoring-demo → Charts**. Select a time range that includes this run. Compare latency, tokens, traffic and cost where pricing is available. Allow a minute or two for ingestion.

## Create an alert

Open **Alerts → Create alert**, select `monitoring-demo`, and choose **LLM response time**. For a 15-minute interval, try a warning above 2,000 ms and a critical threshold above 5,000 ms. Choose an email recipient or your own Slack webhook and save.

## Monitor and manage alerts

Inspect the configuration and trigger history. In a demo project, compare the warning threshold with observed latency and run the traffic cell again. The alert evaluates the window, not a single request.